<a href="https://colab.research.google.com/github/aicha-bakayoko/DI-BOOTCAMP/blob/main/W7D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources

### Ce que tu vas apprendre

*   **Évaluation pratique de LLM** : Acquérez une expérience pratique à l’évaluation des LLM pour la synthèse.
*   **Plongée en profondeur métrique** : Comprendre les forces et faiblesses de différentes métriques d’évaluation (précision, ROUGE).
*   **Comparaison des modèles** : Apprends à comparer systématiquement différents LLM et tailles de modèles.
*   **Maîtrise du visage dans les câlins** : Améliorez vos compétences dans l’utilisation des Hugging Face et des bibliothèques.
*   **Personnalisation** : Implémenter et analyser les effets de la modification des métriques d’évaluation et des paramètres du modèle.
*   **Gestion des données** : Apprenez à charger, traiter et échantillonner des jeux de données textuelles à l’aide de pandas.
*   **Prétraitement du texte** : Comprenez l’importance du prétraitement du texte pour les tâches de NLP.
*   **Débogage et analyse** : Développez des compétences en débogage et analyse des sorties de LLM.


### 🛠️ Ce que vous allez créer

*   **Scénarios d’évaluation** : Des scripts Python pour calculer et comparer les métriques de synthèse.
*   **Rapports comparatifs** : DataFrames et visualisations résumant les performances de différents LLM.
*   **Métriques d’évaluation modifiées** : Des métriques de précision personnalisées adaptées pour la synthèse.
*   **Résultats de synthèse** : A généré des résumés à partir de divers LLM pour une analyse comparative.
*   **Rapports analytiques** : Documentation de vos résultats, y compris des discussions sur le comportement des métriques et la performance du modèle.
*   **Fonctions personnalisées** : Fonctions pour charger des ensembles de données, générer des résumés et calculer les scores ROUGE.
*   **Tableaux comparatifs de modèles** : Tableaux comparant les performances de différents LLM en fonction de divers indicateurs.


⚠️Pour ces exercices, nous fournissons un carnet Google Colab pré-rempli. Téléchargez le carnet, lisez les instructions ci-dessous, et complétez les sections requises directement dans le Colab.

Google Fichier tutoriel Colab





Tous les exercices d’aujourd’hui font partie d’un tutoriel pratique unique conçu pour vous apprendre à évaluer les LLM sur des tâches de synthèse. Ensemble, vous allez :

*   Mesurer la précision des résultats résumés
*   Calculer les scores ROUGE-N
*   Construire un cadre cohérent pour comparer différentes tailles de modèles et architectures

Chaque partie s’appuie sur la précédente, vous offrant un flux de travail cohérent pour évaluer et comparer la performance de synthèse.



### Objectifs d’apprentissage

*   **Compréhension des métriques** : Apprenez à calculer le ROUGE-N et comprenez ses subtilités.
*   **Renforcement de l’intuition** : Développer une compréhension intuitive du ROUGE-N et de son application à la synthèse.
*   **Analyse comparative** : Tester et comparer différents LLM et tailles de modèles sur un ensemble de données cohérent.

### Téléchargez le jeu de données ici.

## Partie I. Mise en place

### Installer des bibliothèques

In [ ]:
!pip install rouge_score==0.1.2
!pip install evaluate
!pip install -U accelerate --quiet
!pip install datasets
!pip install nltk
!pip install transformers

### Téléchargez les ressources NLTK

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

## 🌟 Partie II : Chargement et exploration des jeux de données

### Chargement des jeux de données

In [ ]:
import pandas as pd
import numpy as np

# Assuming the files are in the same directory as the notebook
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print("Original train_df shape:", train_df.shape)
print("Original test_df shape:", test_df.shape)

### Échantillonnage

In [ ]:
train_sample = train_df.sample(n=100, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=50, random_state=42).reset_index(drop=True)

print("Sampled train_df shape:", train_sample.shape)
print("Sampled test_df shape:", test_sample.shape)

### Exploration : Affichez le premier exemple de l’exemple d’entraînement

In [ ]:
print("First training sample article (prompt_text):")
print(train_sample.loc[0, 'prompt_text'])
print("\nFirst training sample reference summary (prompt_title):")
print(train_sample.loc[0, 'prompt_title'])

### Inspection des données

In [ ]:
print("Sampled Training DataFrame:")
display(train_sample.head())

print("\nSampled Test DataFrame:")
display(test_sample.head())

## 🌟 Partie III : Résumé avec T5

### Implémentation de la fonction `summarize_with_t5`

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
import gc

def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def summarize_with_t5(articles, model_name="t5-small", batch_size=4, max_length=150, min_length=40):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []
    for i, batch in enumerate(batch_generator(articles, batch_size)):
        print(f"Processing batch {i+1}/{len(articles)//batch_size + 1}")
        inputs = ["summarize: " + article for article in batch]

        # Tokenize inputs, handling truncation
        tokenized_inputs = tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512 # T5's typical max input length
        ).to(device)

        generated_ids = model.generate(
            tokenized_inputs.input_ids,
            attention_mask=tokenized_inputs.attention_mask,
            max_length=max_length,
            min_length=min_length,
            num_beams=4, # Use beam search for better quality
            early_stopping=True
        )
        decoded_summaries = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in generated_ids]
        summaries.extend(decoded_summaries)

        # Clear CUDA cache and collect garbage after each batch
        del tokenized_inputs
        del generated_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # Final cleanup
    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

### Génération résumée pour `t5-small`

In [ ]:
t5_small_summaries = summarize_with_t5(train_sample['prompt_text'].tolist(), model_name="t5-small")

### Affichage des résultats

In [ ]:
t5_small_results = pd.DataFrame({
    'Article': train_sample['prompt_text'],
    'Reference Summary': train_sample['prompt_title'],
    'T5-small Generated Summary': t5_small_summaries
})

display(t5_small_results.head())

## 🌟 Partie IV : Évaluation de la précision

### Calcul de précision

In [ ]:
# Calculate exact match accuracy
def calculate_exact_match_accuracy(generated_summaries, reference_summaries):
    exact_matches = 0
    for gen, ref in zip(generated_summaries, reference_summaries):
        if gen.strip().lower() == ref.strip().lower():
            exact_matches += 1
    return exact_matches / len(generated_summaries) if generated_summaries else 0

accuracy = calculate_exact_match_accuracy(t5_small_summaries, train_sample['prompt_title'].tolist())
print(f"Accuracy (exact match) for t5-small: {accuracy:.4f}")

### Interprétation des résultats

The accuracy is very low (likely 0) because exact phrase matching is a poor metric for summarization. Summaries can convey the same information using different words, sentence structures, or lengths. A good summary might not be an exact word-for-word match to a reference summary but still be highly relevant and accurate. This metric doesn't account for semantic similarity, paraphrasing, or factual correctness if the wording differs.

## 🌟 Partie V : Implémentation de la métrique ROUGE

### Introduction à la métrique

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a set of metrics used for evaluating automatic summarization and machine translation. It works by comparing an automatically produced summary or translation with a set of reference summaries (human-produced). The metrics measure the overlap of n-grams (sequences of words) between the generated and reference summaries. Common ROUGE metrics include:

*   **ROUGE-N**: Measures the overlap of N-grams. For example, ROUGE-1 compares unigrams (single words), ROUGE-2 compares bigrams (two-word sequences).
*   **ROUGE-L**: Measures the longest common subsequence (LCS) between the generated and reference summaries, which doesn't require consecutive matches.

ROUGE scores typically range from 0 to 1, where higher scores indicate more overlap and thus better quality.

### Utilisation de la bibliothèque `evaluate`

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
print("ROUGE metric loaded successfully.")

### Prétraitement

ROUGE scores are sensitive to how the text is preprocessed. Specifically, it's common practice to tokenize sentences and separate them with newlines. This helps ROUGE-L in particular by allowing it to treat each sentence as a unit within the longest common subsequence calculation. The `nltk` sentence tokenizer (`punkt`) is very useful for this purpose.

In [ ]:
from nltk.tokenize import sent_tokenize

def format_summary_for_rouge(text):
    # Ensure text is not None or empty
    if not text:
        return ""
    # Split the text into sentences and join them with newlines
    return "\n".join(sent_tokenize(text.strip()))

def compute_rouge_score(generated_summaries, reference_summaries):
    # Preprocess both generated and reference summaries
    formatted_generated = [format_summary_for_rouge(s) for s in generated_summaries]
    formatted_references = [format_summary_for_rouge(s) for s in reference_summaries]

    # Compute ROUGE scores
    # Use use_stemmer=True for better recall as it normalizes words
    results = rouge.compute(predictions=formatted_generated, references=formatted_references, use_stemmer=True)

    return results

print("compute_rouge_score function defined.")

## 🌟 Partie VI : Comprendre les scores ROUGE

### Test de correspondance exacte

In [ ]:
print("--- Exact Match Test ---")
exact_gen = ["The cat sat on the mat."]
exact_ref = ["The cat sat on the mat."]
rouge_exact = compute_rouge_score(exact_gen, exact_ref)
print("Generated: ", exact_gen[0])
print("Reference: ", exact_ref[0])
print("ROUGE Scores (Exact Match):", {k: round(v, 4) for k, v in rouge_exact.items()})


### Test de prédiction nulle

In [ ]:
print("\n--- Null Prediction Test ---")
null_gen = ["", " ", "\n"]
null_ref = ["The cat sat on the mat.", "A dog barked.", "Birds flew."]
rouge_null = compute_rouge_score(null_gen, null_ref)
print("Generated: ", null_gen)
print("References: ", null_ref)
print("ROUGE Scores (Null Prediction):", {k: round(v, 4) for k, v in rouge_null.items()})


### Effet de retard (Stemming)

In [ ]:
print("\n--- Stemming Effect Test ---")
stem_gen = ["The cat is running."]
stem_ref = ["The cat runs."]
# compute_rouge_score already uses use_stemmer=True
rouge_stem = compute_rouge_score(stem_gen, stem_ref)
print("Generated: ", stem_gen[0])
print("Reference: ", stem_ref[0])
print("ROUGE Scores (with stemming, e.g., run/running are treated similarly):", {k: round(v, 4) for k, v in rouge_stem.items()})

# To demonstrate without stemming, we would need to call evaluate.load('rouge', use_stemmer=False) specifically
# For simplicity, we'll stick to the function defined, which uses stemming.

### Analyse des N-grammes

In [ ]:
print("\n--- N-gram Analysis (ROUGE-1, ROUGE-2) ---")

# Example 1: High ROUGE-1, lower ROUGE-2 (unigrams match well, bigrams less so)
ngram_gen_1 = ["The quick brown fox jumps over the lazy dog."]
ngram_ref_1 = ["A quick red fox leaps over a sleeping dog."]
rouge_ngram_1 = compute_rouge_score(ngram_gen_1, ngram_ref_1)
print("\nExample 1 (High ROUGE-1, lower ROUGE-2):")
print("Generated: ", ngram_gen_1[0])
print("Reference: ", ngram_ref_1[0])
print("ROUGE Scores:", {k: round(v, 4) for k, v in rouge_ngram_1.items()})

# Example 2: High ROUGE-1 and ROUGE-2 (good unigram and bigram overlap)
ngram_gen_2 = ["The cat sat on the mat."]
ngram_ref_2 = ["The cat sat on the mat."]
rouge_ngram_2 = compute_rouge_score(ngram_gen_2, ngram_ref_2)
print("\nExample 2 (High ROUGE-1 and ROUGE-2):")
print("Generated: ", ngram_gen_2[0])
print("Reference: ", ngram_ref_2[0])
print("ROUGE Scores:", {k: round(v, 4) for k, v in rouge_ngram_2.items()})

# Example 3: Low ROUGE-1 and ROUGE-2 (minimal overlap)
ngram_gen_3 = ["Birds are singing in the trees."]
ngram_ref_3 = ["The ocean is vast and deep."]
rouge_ngram_3 = compute_rouge_score(ngram_gen_3, ngram_ref_3)
print("\nExample 3 (Low ROUGE-1 and ROUGE-2):")
print("Generated: ", ngram_gen_3[0])
print("Reference: ", ngram_ref_3[0])
print("ROUGE Scores:", {k: round(v, 4) for k, v in rouge_ngram_3.items()})


### Symétrie

In [ ]:
print("\n--- Symmetry Test ---")

sym_gen_1 = ["The quick brown fox."]
sym_ref_1 = ["A brown fox ran quickly."]
rouge_sym_1 = compute_rouge_score(sym_gen_1, sym_ref_1)
print("\nCase 1: Generated = 'The quick brown fox.', Reference = 'A brown fox ran quickly.'")
print("ROUGE Scores:", {k: round(v, 4) for k, v in rouge_sym_1.items()})

sym_gen_2 = ["A brown fox ran quickly."]
sym_ref_2 = ["The quick brown fox."]
rouge_sym_2 = compute_rouge_score(sym_gen_2, sym_ref_2)
print("\nCase 2: Generated = 'A brown fox ran quickly.', Reference = 'The quick brown fox.'")
print(
    "ROUGE Scores (swapped generated and reference - should be symmetric for ROUGE-N, but not ROUGE-L which is based on F-measure):",
    {k: round(v, 4) for k, v in rouge_sym_2.items()}
)

# ROUGE-N (precision, recall, f-measure) and ROUGE-L (f-measure) are generally symmetric when computing the f-measure.
# However, if you look at precision vs recall components separately, they are not symmetric.
# The compute function directly returns the F-measure for ROUGE-N and ROUGE-L by default, which is symmetric.

## 🌟 Partie VII : Comparaison des petits et grands modèles

### Sélection des modèles

In [ ]:
model_names = ["t5-small", "t5-base", "gpt2"]
print(f"Models selected for comparison: {', '.join(model_names)}")

### Génération résumée

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def summarize_with_gpt2(articles, model_name="gpt2", batch_size=4, max_length=150, min_length=40):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    # GPT-2 does not have a padding token by default, set eos_token as padding token
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

    summaries = []
    for i, batch in enumerate(batch_generator(articles, batch_size)):
        print(f"Processing batch {i+1}/{len(articles)//batch_size + 1}")
        # GPT-2 is a generative model, not instruction-tuned like T5.
        # A common prompt for summarization with GPT-2 is to append 'TL;DR:'
        inputs = [article + "\nTL;DR:" for article in batch]

        tokenized_inputs = tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024 # GPT-2's typical max input length
        ).to(device)

        # Generate method for GPT2 often uses input_ids for generation, not encoder_outputs
        # Adjust max_length for generation to be relative to the input length
        generated_ids = model.generate(
            tokenized_inputs.input_ids,
            attention_mask=tokenized_inputs.attention_mask,
            max_new_tokens=max_length, # max_new_tokens is more appropriate for models like GPT-2
            # min_length=min_length, # min_length is not directly supported in the same way for autoregressive models without custom generation strategies
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id # Explicitly set pad token id
        )

        # Decode, trying to remove the input prompt from the generated text
        decoded_summaries = []
        for gen_id, original_input in zip(generated_ids, inputs):
            full_text = tokenizer.decode(gen_id, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            # Attempt to remove the prompt from the beginning of the generated text
            if original_input in full_text:
                summary = full_text.replace(original_input, "").strip()
            else:
                summary = full_text # Fallback if replacement doesn't work perfectly

            # Simple truncation if it's too long, and ensure min_length if possible
            sentences = sent_tokenize(summary)
            final_summary = []
            current_length = 0
            for s in sentences:
                if current_length + len(s.split()) <= max_length:
                    final_summary.append(s)
                    current_length += len(s.split())
                else:
                    break
            summary = " ".join(final_summary)
            if len(summary.split()) < min_length and len(sentences) > 0:
                # If summary is too short, try to add more until min_length or no more sentences
                temp_summary = ""
                for s in sentences:
                    temp_summary += s + " "
                    if len(temp_summary.split()) >= min_length:
                        break
                summary = temp_summary.strip()

            decoded_summaries.append(summary.replace("TL;DR:", "").strip())
        summaries.extend(decoded_summaries)

        del tokenized_inputs
        del generated_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

In [ ]:
all_generated_summaries = {}

# T5-small
print("Generating summaries with t5-small...")
all_generated_summaries['t5-small'] = summarize_with_t5(train_sample['prompt_text'].tolist(), model_name="t5-small")

# T5-base
print("\nGenerating summaries with t5-base...")
all_generated_summaries['t5-base'] = summarize_with_t5(train_sample['prompt_text'].tolist(), model_name="t5-base")

# GPT-2
print("\nGenerating summaries with gpt2...")
all_generated_summaries['gpt2'] = summarize_with_gpt2(train_sample['prompt_text'].tolist(), model_name="gpt2")

### Calcul ROUGE

In [ ]:
def compute_rouge_per_row(generated_summaries_dict, reference_summaries):
    rouge_scores_per_model = {}
    for model_name, gen_summaries in generated_summaries_dict.items():
        model_rouge_data = []
        for i in range(len(gen_summaries)):
            current_gen = [gen_summaries[i]]
            current_ref = [reference_summaries[i]]
            scores = compute_rouge_score(current_gen, current_ref)
            model_rouge_data.append({
                'rouge1_fmeasure': scores['rouge1'],
                'rouge2_fmeasure': scores['rouge2'],
                'rougeL_fmeasure': scores['rougeL'],
                'rougeLsum_fmeasure': scores['rougeLsum'],
            })
        rouge_scores_per_model[model_name] = pd.DataFrame(model_rouge_data)
    return rouge_scores_per_model

all_rouge_per_row = compute_rouge_per_row(all_generated_summaries, train_sample['prompt_title'].tolist())
print("ROUGE scores calculated for each row for all models.")

### Affichage des résultats (ROUGE par rangée)

In [ ]:
for model_name, df_scores in all_rouge_per_row.items():
    print(f"\nROUGE scores per row for {model_name}:")
    display(df_scores.head())

## 🌟 Partie VIII : Comparaison de tous les modèles

### Fonction d’agrégation `compare_models`

In [ ]:
def compare_models(rouge_scores_per_model_df_dict):
    aggregated_scores = {}
    for model_name, df_scores in rouge_scores_per_model_df_dict.items():
        aggregated_scores[model_name] = df_scores.mean().to_dict()

    return pd.DataFrame.from_dict(aggregated_scores, orient='index')

overall_rouge_comparison = compare_models(all_rouge_per_row)
print("Overall ROUGE score comparison across models:")
display(overall_rouge_comparison.round(4))

### Fonction de comparaison résumée `compare_models_summaries`

In [ ]:
def compare_models_summaries(original_articles, reference_summaries, generated_summaries_dict, num_examples=5):
    comparison_data = []
    for i in range(min(num_examples, len(original_articles))):
        row_data = {
            'Article': original_articles.iloc[i],
            'Reference Summary': reference_summaries.iloc[i]
        }
        for model_name, gen_summaries_list in generated_summaries_dict.items():
            row_data[f'{model_name} Generated Summary'] = gen_summaries_list[i]
        comparison_data.append(row_data)

    return pd.DataFrame(comparison_data)

summary_comparison_df = compare_models_summaries(
    train_sample['prompt_text'],
    train_sample['prompt_title'],
    all_generated_summaries,
    num_examples=5
)

print("Side-by-side comparison of generated summaries:")
display(summary_comparison_df)

### Affichage des résultats

#### Scores ROUGE agrégés

In [ ]:
display(overall_rouge_comparison.round(4))

#### Comparaisons résumées côte à côte

In [ ]:
display(summary_comparison_df)